<a href="https://colab.research.google.com/github/HArmor28/ML/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HArmor28/ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Rule: Prioritize content for a refresh review when it has not been updated recently but still receives meaningful search impressions. Older content with more observed impressions should rank higher because it represents a larger measureable opportunity. This score is directional decision support; it does not prove that refreshing the content will improve performance.

Reason code: STALE_BUT_VISIBLE: the content has gone a long time without an update and still receives enough search impressions to justify review.

Action label: REVIEW_FOR_REFRESH: review the content and decide whether its information structure or search presentation needs updating.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from pathlib import Path

# Load the public starter dataset
data_url = (
    "https://raw.githubusercontent.com/"
    "HArmor28/ML/main/data/raw/content_refresh_anonymized.csv"
)
df = pd.read_csv(data_url)

# Transparent rule thresholds
STALE_DAYS = 181
MIN_IMPRESSIONS = 300

# Keep pages that are both stale and meaningfully visible
qualifies = (
    (df["days_since_last_update"] >= STALE_DAYS)
    & (df["impressions_90d"] >= MIN_IMPRESSIONS)
)

queue = df.loc[
    qualifies,
    ["content_id", "days_since_last_update", "impressions_90d"],
].copy()

# Older and more visible pages receive higher scores
queue["score"] = (
    queue["days_since_last_update"]
    * queue["impressions_90d"]
)

queue["reason_code"] = "STALE_BUT_VISIBLE"
queue["action_label"] = "REVIEW_FOR_REFRESH"

# Rank highest score first
queue = queue.sort_values(
    ["score", "impressions_90d", "content_id"],
    ascending=[False, False, True],
).reset_index(drop=True)

queue["rank"] = range(1, len(queue) + 1)

# Put columns in a readable order
queue = queue[
    [
        "rank",
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "score",
        "reason_code",
        "action_label",
    ]
]

# Write the required output
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)

# Honest checks
assert queue["rank"].is_unique
assert queue["score"].notna().all()
assert (queue["reason_code"] == "STALE_BUT_VISIBLE").all()
assert (queue["action_label"] == "REVIEW_FOR_REFRESH").all()
assert queue["score"].is_monotonic_decreasing

print(f"Rows in source data: {len(df):,}")
print(f"Pages qualifying for review: {len(queue):,}")
print(f"CSV written to: {output_path}")
display(queue.head(20))

Rows in source data: 30,000
Pages qualifying for review: 22
CSV written to: work/outputs/baseline_action_score.csv


,rank,content_id,days_since_last_update,impressions_90d,score,reason_code,action_label
0,1,content_cf56e2e2e282,194,61678,11965532,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
1,2,content_7368877ea310,194,59472,11537568,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
2,3,content_1bfaa38ff26c,194,25715,4988710,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
3,4,content_0a91db491d14,193,13299,2566707,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
4,5,content_5feee3994adb,194,7812,1515528,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
5,6,content_c2d929d83eaa,193,7558,1458694,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
6,7,content_b16bd7307b39,194,4590,890460,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
7,8,content_fe16a55cd13d,194,4556,883864,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
8,9,content_ecb6215e79fd,194,4429,859226,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH
9,10,content_928af3e22c80,193,1697,327521,STALE_BUT_VISIBLE,REVIEW_FOR_REFRESH


## 3. Top-20 review

I reviewed the top 20 as directional candidates for REVIEW_FOR_REFRESH, all with reason code STALE_BUT_VISIBLE. Confidence is higher when a page clears both thresholds by a wide margin. Confidence is lower when its observed days or impressions are close to the cutoff. Every recommendation could be wrong if the update date is inaccurate, the impressions are temporary or seasonal, the page has already been reviewed, or the content is intentionally evergreen.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20_review = queue.head(20).copy()

# Boundary cases receive lower confidence because small measurement
# changes could remove them from the queue.
top20_review["near_stale_cutoff"] = (
    top20_review["days_since_last_update"] <= STALE_DAYS + 7
)
top20_review["near_impressions_cutoff"] = (
    top20_review["impressions_90d"] <= MIN_IMPRESSIONS * 1.25
)

def confidence_note(row):
    if row["near_stale_cutoff"] and row["near_impressions_cutoff"]:
        return "Low: both measured inputs are close to the rule thresholds."
    if row["near_stale_cutoff"]:
        return "Medium: visibility is measured, but staleness is close to the cutoff."
    if row["near_impressions_cutoff"]:
        return "Medium: staleness is measured, but visibility is close to the cutoff."
    return "High for review priority: both measured inputs clear the thresholds."

def wrong_condition(row):
    conditions = [
        "the update date is inaccurate",
        "the observed impressions are temporary or seasonal",
        "the page was already reviewed or is intentionally evergreen",
    ]
    if row["near_stale_cutoff"]:
        conditions.append("a small date change puts it below the stale threshold")
    if row["near_impressions_cutoff"]:
        conditions.append("a small impressions change puts it below the visibility threshold")
    return "; ".join(conditions) + "."

top20_review["confidence_note"] = top20_review.apply(
    confidence_note, axis=1
)
top20_review["what_would_make_it_wrong"] = top20_review.apply(
    wrong_condition, axis=1
)

review_columns = [
    "rank",
    "content_id",
    "action_label",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong",
]

assert len(top20_review) == 20
assert top20_review[review_columns].notna().all().all()

display(top20_review[review_columns])

,rank,content_id,action_label,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,High for review priority: both measured inputs...,the update date is inaccurate; the observed im...
1,2,content_7368877ea310,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,High for review priority: both measured inputs...,the update date is inaccurate; the observed im...
2,3,content_1bfaa38ff26c,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,High for review priority: both measured inputs...,the update date is inaccurate; the observed im...
3,4,content_0a91db491d14,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,High for review priority: both measured inputs...,the update date is inaccurate; the observed im...
4,5,content_5feee3994adb,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,High for review priority: both measured inputs...,the update date is inaccurate; the observed im...
5,6,content_c2d929d83eaa,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,High for review priority: both measured inputs...,the update date is inaccurate; the observed im...
6,7,content_b16bd7307b39,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,High for review priority: both measured inputs...,the update date is inaccurate; the observed im...
7,8,content_fe16a55cd13d,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,High for review priority: both measured inputs...,the update date is inaccurate; the observed im...
8,9,content_ecb6215e79fd,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,High for review priority: both measured inputs...,the update date is inaccurate; the observed im...
9,10,content_928af3e22c80,REVIEW_FOR_REFRESH,STALE_BUT_VISIBLE,High for review priority: both measured inputs...,the update date is inaccurate; the observed im...


## 4. Weak picks + leakage check

The weakest picks are the threshold-adjacent candidates. Ranks 16 and 19 have only 335 and 304 observed impressions, so small measurement changes could make them ineligible. Ranks 12, 17, 18, and 20 have only 183 measured stale days, two days above the cutoff. These are still reasonable review candidates, but the evidence is weaker.
The score uses only days_since_last_update and impressions_90d. It does not use product flags, manual priority fields, labels, outcomes, post-action measurements, or future windows. Therefore, no product or future information leaked into this baseline.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
weak_picks = top20_review.loc[
    top20_review["near_stale_cutoff"]
    | top20_review["near_impressions_cutoff"]
].copy()

def weakness_reason(row):
    reasons = []
    if row["near_stale_cutoff"]:
        reasons.append(
            f'{row["days_since_last_update"]} days is close to '
            f"the {STALE_DAYS}-day cutoff"
        )
    if row["near_impressions_cutoff"]:
        reasons.append(
            f'{row["impressions_90d"]} impressions is close to '
            f"the {MIN_IMPRESSIONS}-impression cutoff"
        )
    return "; ".join(reasons)

weak_picks["why_weak"] = weak_picks.apply(weakness_reason, axis=1)

# Leakage audit: these are the only source columns used by the score.
score_inputs = {
    "days_since_last_update",
    "impressions_90d",
}

# Flag source fields that appear to describe product decisions,
# labels, outcomes, or measurements from the future.
leakage_markers = (
    "product",
    "flag",
    "priority",
    "manual",
    "label",
    "target",
    "outcome",
    "future",
    "next_",
    "after_",
    "post_",
)

suspicious_source_columns = {
    column
    for column in df.columns
    if any(marker in column.lower() for marker in leakage_markers)
}

used_suspicious_columns = score_inputs & suspicious_source_columns

assert score_inputs == {
    "days_since_last_update",
    "impressions_90d",
}
assert not used_suspicious_columns

print("Score inputs:", sorted(score_inputs))
print("Suspicious source fields not used:", sorted(suspicious_source_columns))
print("Leakage found in score:", sorted(used_suspicious_columns))
display(
    weak_picks[
        [
            "rank",
            "content_id",
            "days_since_last_update",
            "impressions_90d",
            "why_weak",
        ]
    ]
)

Score inputs: ['days_since_last_update', 'impressions_90d']
Suspicious source fields not used: []
Leakage found in score: []


,rank,content_id,days_since_last_update,impressions_90d,why_weak
11,12,content_e3ff1b093148,183,1408,183 days is close to the 181-day cutoff
15,16,content_4729b57ca036,301,335,335 impressions is close to the 300-impression...
16,17,content_6226ee6adc91,183,545,183 days is close to the 181-day cutoff
17,18,content_074ba6ead17b,183,533,183 days is close to the 181-day cutoff
18,19,content_6476d1d8c050,313,304,304 impressions is close to the 300-impression...
19,20,content_fd16e3475c29,183,429,183 days is close to the 181-day cutoff


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.